# envs

> Circular heading arena and base world class for explRL agents.


In [ ]:
#| default_exp envs


In [ ]:
#| hide
from nbdev.showdoc import *


In [ ]:
#| export
import numpy as np


In [ ]:
#| export
class World:
    def __init__(self, size=None):
        self.size = size


In [ ]:
#| export
class HeadingArena(World):
    '''Circular arena with a goal heading and Eq. 19 reward.'''
    def __init__(self, goal: float = np.pi):
        super().__init__(2 * np.pi)
        self.goal = float(goal) % (2 * np.pi)

    def distance(self, theta) -> float:
        'Wrapped distance to the goal in radians, in [0, pi].'
        return abs((float(theta) - self.goal + np.pi) % (2 * np.pi) - np.pi)

    def reward(self, theta) -> float:
        return (np.pi / 10) * (0.5 - self.distance(theta))

    def step(self, theta: float, dx: float) -> tuple:
        new = (float(theta) + float(dx)) % (2 * np.pi)
        return new, self.reward(new)


In [ ]:
#| export
class ZoneArena(World):
    '''Arena with safe/danger zones on an N-step heading grid (degrees at centers).'''
    def __init__(self, N=32, symmetric=True, centers=(90, 270), width=90):
        super().__init__(N)
        self.N = int(N)
        self.symmetric = bool(symmetric)
        self.centers = tuple(float(c) for c in centers)
        self.width = float(width)
        self._step_deg = 360.0 / self.N

    def _deg(self, a):
        return (int(a) % self.N) * self._step_deg

    def isSafe(self, a) -> bool:
        'True if grid heading a lies in [center - width/2, center + width/2) for any center.'
        d = self._deg(a)
        half = self.width / 2
        for c in self.centers:
            lo, hi = c - half, c + half
            if lo <= d < hi:
                return True
        return False

    def valence(self, a) -> int:
        'Eq. 40: +1 in a safe zone, -1 in a danger zone.'
        return 1 if self.isSafe(a) else -1

    def ringActivity(self, a):
        'Ring-neuron activity seen at grid heading a in this scene (one or two bumps).'
        from explRL.circuit import ringActivity
        return ringActivity(a, self.N, self.symmetric)

    def safeDistance(self, theta) -> float:
        'Degrees from arena angle theta (radians) to the nearest safe-zone center.'
        deg = float(np.rad2deg(theta) % 360)
        return float(min(min(abs(deg - c), 360 - abs(deg - c)) for c in self.centers))


In [ ]:
from explRL.circuit import ringActivity
from fastcore.test import test_eq, test_close

za = ZoneArena()
safe = sum(za.isSafe(a) for a in range(32))
test_eq(safe, 16)
test_eq(za.isSafe(8), True)   # 90 deg
test_eq(za.isSafe(24), True)  # 270 deg
test_eq(za.isSafe(0), False)  # 0 deg danger
test_eq(za.isSafe(16), False) # 180 deg danger
test_eq(za.isSafe(4), True)   # 45 deg safe (half-open at 135)
test_eq(za.isSafe(12), False) # 135 deg danger
test_eq(za.valence(8), 1)
test_eq(za.valence(0), -1)
test_close(za.ringActivity(3), ringActivity(3, 32, True))
test_close(za.ringActivity(3), ringActivity(3, 32, za.symmetric))
zas = ZoneArena(symmetric=False)
test_close(zas.ringActivity(5), ringActivity(5, 32, False))
d0 = za.safeDistance(za._step_deg * np.pi / 180 * 8)
d1 = za.safeDistance(za._step_deg * np.pi / 180 * 24)
test_close(d0, 0.0)
test_close(za.safeDistance(np.deg2rad(0)), 90.0)
# 180 deg apart same distance to safe centers
test_close(za.safeDistance(np.deg2rad(10)), za.safeDistance(np.deg2rad(190)))



In [ ]:
from math import pi
from fastcore.test import test_eq, test_close

env = HeadingArena(goal=pi)
test_eq(env.reward(pi), pi / 20)
test_close(env.reward(0.0), (pi / 10) * (0.5 - pi))


In [ ]:
d = 0.5
test_close(env.reward(env.goal + d), env.reward(env.goal - d))


In [ ]:
new, r = env.step(2 * pi - 0.1, 0.2)
test_close(new, 0.1)
test_close(r, env.reward(0.1))


In [ ]:
new, _ = env.step(0.1, -0.2)
test_close(new, 2 * pi - 0.1)


In [ ]:
e = HeadingArena(goal=-pi / 2)
test_close(e.goal, 3 * pi / 2)


In [ ]:
test_eq(World().size, None)
test_eq(World(7).size, 7)
test_eq(HeadingArena().size, 2 * pi)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()
